In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_json, from_json, current_timestamp, get_json_object, explode, first, explode_outer
from pyspark.sql.types import StructType, StructField, StringType, ArrayType, MapType
import os

In [2]:
try:
    spark.stop()
except:
    pass

In [3]:
spark = (SparkSession.builder.appName("HealthcareDataProcessing")
.config("spark.sql.files.ignoreCorruptFiles", "true")
.config("spark.driver.memory", "4g") 
.config("spark.executor.memory", "4g") 
.config("spark.memory.offHeap.enabled", "true") 
.config("spark.memory.offHeap.size", "2g") 
.master("local[*]")
.getOrCreate())

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/08 12:23:39 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
bronze_path = "../../data_lake/bronze/batch_data/resource_type=Patient/"
silver_base_path = "../../data_lake/silver/silver_Patient/"

In [5]:
#schema for patient resource

communication_schema = ArrayType(StructType([
    StructField("language", StructType([
        StructField("coding", ArrayType(StructType([
            StructField("system", StringType(), True),
            StructField("code", StringType(), True),
            StructField("display", StringType(), True)
        ])), True),
        StructField("text", StringType(), True)
    ]), True),
    StructField("preferred", StringType(), True)
    
]))

#Marital schema

marital_schema = (StructType([
    StructField("coding", ArrayType(StructType([
        StructField("system", StringType(), True),
        StructField("code", StringType(), True),
        StructField("display", StringType(), True)
    ])), True),
    StructField("text", StringType(), True)
]))



In [6]:
df_patient = spark.read.format("parquet").load(bronze_path)
df_patient.printSchema()

root
 |-- bundle_resource_type: string (nullable = true)
 |-- bundle_type: string (nullable = true)
 |-- fullUrl: string (nullable = true)
 |-- resource: map (nullable = true)
 |    |-- key: string
 |    |-- value: string (valueContainsNull = true)
 |-- request: map (nullable = true)
 |    |-- key: string
 |    |-- value: string (valueContainsNull = true)
 |-- input_file_name: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)



In [8]:
df_intermediate_1 = df_patient.select(
    col("*"),
    col("resource.id").alias("patient_id"),
    col("resource.meta").alias("patient_meta"),
    col("resource.text").alias("patient_text"),
    col("resource.extension").alias("extension"),
    col("resource.identifier").alias("identifier"),
    col("resource.name").alias("patient_name"),
    col("resource.telecom").alias("patient_telecom"),
    col("resource.gender").alias("gender"),
    col("resource.birthDate").alias("birthDate"),
    col("resource.deceasedDateTime").alias("deceasedDateTime"),
    col("resource.address").alias("address"),
    col("resource.maritalStatus").alias("maritalStatus"),
    col("resource.multipleBirthBoolean").alias("multipleBirthBoolean"),
    col("resource.communication").alias("communication")
).drop("resource")

In [11]:
df_intermediate_2 = (df_intermediate_1
                     .withColumn("communication", from_json(col("communication"), communication_schema))
                     .withColumn("patient_language_code", col("communication")[0]["language"]["coding"][0]["code"])
                     .withColumn("patient_language", col("communication")[0]["language"]["coding"][0]["display"])
                     .withColumn("maritalStatus_parsed", from_json(col("maritalStatus"), marital_schema))
                        .withColumn("marital_status_code", col("maritalStatus_parsed")["coding"][0]["code"])
                        .withColumn("marital_status_display", col("maritalStatus_parsed")["coding"][0]["display"])
).drop("maritalStatus_parsed","maritalStatus")           

In [ ]:
df_intermediate_2.write.mode("append").parquet(silver_base_path)